# Module 4.3: Integrated Fundamental Model

## Learning Objectives
By completing this notebook, you will:
1. Build unified fundamental models spanning multiple commodities
2. Model cross-commodity relationships (oil production → associated gas)
3. Create LLM-powered fundamental dashboards
4. Generate portfolio-level fundamental assessments

## Prerequisites
- Completion of Module 4.1 (Crude Fundamentals)
- Completion of Module 4.2 (Natural Gas Balance)
- Understanding of commodity market interactions

## Estimated Time: 120 minutes

---

## Setup & Imports

In [ ]:
# Purpose: Import libraries and load previous models
# Key Concept: Building on crude and gas models for integrated view

import os
import json
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass, asdict
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from enum import Enum

from langchain_anthropic import ChatAnthropic
from langchain.prompts import ChatPromptTemplate
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
np.random.seed(42)

print("✓ Imports complete")

## 1. Understanding Cross-Commodity Relationships

### Why Integration Matters

Commodity markets don't exist in isolation. Key relationships:

**Oil → Natural Gas:**
- Associated gas from oil production (Permian Basin)
- Substitution in power generation
- Rig count and drilling economics

**Natural Gas → Power:**
- Gas-fired generation competes with coal
- Renewable penetration affects residual demand

**Crude Products → Fundamentals:**
- Refinery runs drive crude demand
- Crack spreads influence refinery utilization

### Integration Benefits

1. **Consistency**: Cross-validate fundamentals across markets
2. **Leading Indicators**: Oil production changes → gas supply shifts
3. **Portfolio View**: Unified risk assessment
4. **Arbitrage Opportunities**: Identify relative value

## 2. Integrated Fundamental Data Model

In [ ]:
# Purpose: Define unified multi-commodity fundamental structure
# Key Concept: Portfolio-level fundamental representation

class CommodityType(str, Enum):
    """Commodity types in the integrated model."""
    CRUDE_OIL = "crude_oil"
    NATURAL_GAS = "natural_gas"
    GASOLINE = "gasoline"
    DIESEL = "diesel"


class MarketBalance(str, Enum):
    """Market balance characterization."""
    TIGHT = "tight"
    BALANCED = "balanced"
    OVERSUPPLIED = "oversupplied"


class CommodityFundamental(BaseModel):
    """Fundamental snapshot for a single commodity."""
    
    commodity: CommodityType
    date: str
    supply: float = Field(description="Total supply (commodity-specific units)")
    demand: float = Field(description="Total demand")
    balance: float = Field(description="Supply minus demand")
    inventory: float = Field(description="Current inventory level")
    inventory_change: float = Field(description="Weekly/monthly change")
    vs_avg_pct: float = Field(description="Inventory vs historical average (%)")
    days_supply: float = Field(description="Days of inventory coverage")
    market_balance: MarketBalance
    

class CrossCommodityRelationship(BaseModel):
    """Relationship between two commodities."""
    
    source: CommodityType
    target: CommodityType
    relationship_type: str = Field(description="e.g., 'associated_production', 'substitution'")
    strength: float = Field(description="Relationship strength (0-1)", ge=0, le=1)
    description: str


class IntegratedFundamentals(BaseModel):
    """Integrated multi-commodity fundamental view."""
    
    date: str
    commodities: List[CommodityFundamental]
    relationships: List[CrossCommodityRelationship]
    overall_assessment: str
    key_themes: List[str]
    risks: List[str]


print("✓ Integrated fundamental models defined")

## 3. Building the Integrated Model

In [ ]:
# Purpose: Construct integrated fundamental model from individual commodity data
# Key Concept: Synthesizing multi-market fundamentals

class IntegratedFundamentalModel:
    """Build and analyze integrated commodity fundamentals."""
    
    def __init__(self):
        self.commodities: Dict[CommodityType, CommodityFundamental] = {}
        self.relationships: List[CrossCommodityRelationship] = []
        
    def add_commodity(self, fundamental: CommodityFundamental):
        """Add commodity fundamental data to the model."""
        self.commodities[fundamental.commodity] = fundamental
        
    def add_relationship(self, relationship: CrossCommodityRelationship):
        """Define cross-commodity relationship."""
        self.relationships.append(relationship)
        
    def calculate_associated_gas(self, oil_production: float, 
                                gas_oil_ratio: float = 1.2) -> float:
        """
        Calculate associated natural gas from oil production.
        
        Parameters
        ----------
        oil_production : float
            Crude oil production (million b/d)
        gas_oil_ratio : float
            Associated gas ratio (Mcf per barrel)
            
        Returns
        -------
        float
            Associated gas production (Bcf/d)
        """
        # Convert: million b/d * Mcf/bbl * 1000 / 1e6 = Bcf/d
        associated_gas = oil_production * gas_oil_ratio * 1000 / 1e6
        return associated_gas
    
    def analyze_substitution(self, oil_price: float, gas_price: float,
                            oil_to_gas_ratio: float = 6.0) -> Dict[str, float]:
        """
        Analyze oil-gas substitution economics.
        
        Parameters
        ----------
        oil_price : float
            Oil price ($/barrel)
        gas_price : float
            Gas price ($/MMBtu)
        oil_to_gas_ratio : float
            Typical oil-to-gas energy ratio
            
        Returns
        -------
        dict
            Substitution analysis
        """
        # Calculate oil price in $/MMBtu equivalent
        oil_equivalent = oil_price / oil_to_gas_ratio
        
        # Gas advantage (positive = gas cheaper)
        gas_advantage = oil_equivalent - gas_price
        gas_advantage_pct = (gas_advantage / oil_equivalent) * 100
        
        return {
            'oil_equivalent_mmbtu': oil_equivalent,
            'gas_price': gas_price,
            'gas_advantage': gas_advantage,
            'gas_advantage_pct': gas_advantage_pct,
            'gas_competitive': gas_advantage > 0
        }
    
    def get_summary(self) -> Dict:
        """Generate integrated summary."""
        summary = {
            'date': datetime.now().strftime('%Y-%m-%d'),
            'commodity_count': len(self.commodities),
            'relationship_count': len(self.relationships)
        }
        
        # Aggregate balances
        balances = [c.market_balance for c in self.commodities.values()]
        balance_counts = {b: balances.count(b) for b in set(balances)}
        summary['balance_distribution'] = balance_counts
        
        # Average days of supply
        avg_days_supply = np.mean([c.days_supply for c in self.commodities.values()])
        summary['avg_days_supply'] = avg_days_supply
        
        return summary


# Initialize model
integrated_model = IntegratedFundamentalModel()

print("✓ Integrated model initialized")

## 4. Loading Multi-Commodity Data

In [ ]:
# Purpose: Create synthetic multi-commodity fundamental dataset
# Key Concept: Realistic commodity fundamental scenarios

def create_commodity_fundamentals() -> List[CommodityFundamental]:
    """
    Generate synthetic fundamental data for multiple commodities.
    In production, this would pull from EIA APIs and other data sources.
    """
    fundamentals = []
    
    # Crude Oil
    crude = CommodityFundamental(
        commodity=CommodityType.CRUDE_OIL,
        date="2023-12-22",
        supply=13.3,  # million b/d
        demand=20.8,  # million b/d (includes exports)
        balance=-7.5,
        inventory=439.7,  # million barrels
        inventory_change=8.9,
        vs_avg_pct=-4.0,
        days_supply=21.1,
        market_balance=MarketBalance.BALANCED
    )
    fundamentals.append(crude)
    
    # Natural Gas
    nat_gas = CommodityFundamental(
        commodity=CommodityType.NATURAL_GAS,
        date="2023-12-22",
        supply=104.5,  # Bcf/d
        demand=126.8,  # Bcf/d
        balance=-22.3,
        inventory=3522,  # Bcf
        inventory_change=-112,
        vs_avg_pct=10.2,
        days_supply=27.8,
        market_balance=MarketBalance.OVERSUPPLIED
    )
    fundamentals.append(nat_gas)
    
    # Gasoline
    gasoline = CommodityFundamental(
        commodity=CommodityType.GASOLINE,
        date="2023-12-22",
        supply=10.2,  # million b/d
        demand=8.8,  # million b/d
        balance=1.4,
        inventory=225,  # million barrels
        inventory_change=5.2,
        vs_avg_pct=2.5,
        days_supply=25.6,
        market_balance=MarketBalance.BALANCED
    )
    fundamentals.append(gasoline)
    
    # Diesel
    diesel = CommodityFundamental(
        commodity=CommodityType.DIESEL,
        date="2023-12-22",
        supply=5.1,  # million b/d
        demand=4.2,  # million b/d
        balance=0.9,
        inventory=115,  # million barrels
        inventory_change=-2.1,
        vs_avg_pct=-8.5,
        days_supply=27.4,
        market_balance=MarketBalance.TIGHT
    )
    fundamentals.append(diesel)
    
    return fundamentals


# Load commodity fundamentals
commodity_data = create_commodity_fundamentals()

# Add to integrated model
for commodity in commodity_data:
    integrated_model.add_commodity(commodity)

print(f"✓ Loaded {len(commodity_data)} commodities")
for c in commodity_data:
    print(f"  - {c.commodity.value}: {c.market_balance.value} market")

## 5. Modeling Cross-Commodity Relationships

In [ ]:
# Purpose: Define and analyze cross-commodity relationships
# Key Concept: Quantifying market interdependencies

# Define relationships
relationships = [
    CrossCommodityRelationship(
        source=CommodityType.CRUDE_OIL,
        target=CommodityType.NATURAL_GAS,
        relationship_type="associated_production",
        strength=0.6,
        description="Permian oil production generates associated natural gas (~1.2 Mcf/bbl)"
    ),
    CrossCommodityRelationship(
        source=CommodityType.CRUDE_OIL,
        target=CommodityType.GASOLINE,
        relationship_type="refining_output",
        strength=0.9,
        description="Crude is refined into gasoline (~45% yield)"
    ),
    CrossCommodityRelationship(
        source=CommodityType.CRUDE_OIL,
        target=CommodityType.DIESEL,
        relationship_type="refining_output",
        strength=0.9,
        description="Crude is refined into diesel (~30% yield)"
    ),
    CrossCommodityRelationship(
        source=CommodityType.NATURAL_GAS,
        target=CommodityType.CRUDE_OIL,
        relationship_type="substitution",
        strength=0.4,
        description="Gas can substitute for oil in power generation and some industrial uses"
    )
]

# Add to model
for rel in relationships:
    integrated_model.add_relationship(rel)

print(f"✓ Defined {len(relationships)} cross-commodity relationships")
for rel in relationships:
    print(f"  {rel.source.value} → {rel.target.value}: {rel.relationship_type} (strength: {rel.strength})")

In [ ]:
# Purpose: Calculate cross-commodity impacts
# Key Concept: Quantifying spillover effects

# Calculate associated gas from oil production
crude_fundamental = integrated_model.commodities[CommodityType.CRUDE_OIL]
associated_gas = integrated_model.calculate_associated_gas(
    oil_production=crude_fundamental.supply,
    gas_oil_ratio=1.2
)

gas_fundamental = integrated_model.commodities[CommodityType.NATURAL_GAS]
associated_pct = (associated_gas / gas_fundamental.supply) * 100

print("=== ASSOCIATED GAS ANALYSIS ===")
print(f"Oil Production: {crude_fundamental.supply:.1f} million b/d")
print(f"Associated Gas: {associated_gas:.1f} Bcf/d")
print(f"Total Gas Supply: {gas_fundamental.supply:.1f} Bcf/d")
print(f"Associated Gas Share: {associated_pct:.1f}%")

# Analyze oil-gas substitution
oil_price = 72.0  # $/barrel
gas_price = 2.5   # $/MMBtu

substitution = integrated_model.analyze_substitution(oil_price, gas_price)

print("\n=== OIL-GAS SUBSTITUTION ECONOMICS ===")
print(f"Oil Price: ${oil_price:.2f}/bbl")
print(f"Gas Price: ${gas_price:.2f}/MMBtu")
print(f"Oil Equivalent: ${substitution['oil_equivalent_mmbtu']:.2f}/MMBtu")
print(f"Gas Advantage: ${substitution['gas_advantage']:.2f}/MMBtu ({substitution['gas_advantage_pct']:.1f}%)")
print(f"Gas Competitive: {'YES' if substitution['gas_competitive'] else 'NO'}")

## 6. Integrated Fundamental Dashboard

In [ ]:
# Purpose: Create comprehensive multi-commodity visualization
# Key Concept: Portfolio-level fundamental overview

def create_integrated_dashboard(model: IntegratedFundamentalModel):
    """
    Generate integrated fundamental dashboard.
    
    Parameters
    ----------
    model : IntegratedFundamentalModel
        Populated integrated model
    """
    fig = plt.figure(figsize=(16, 12))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
    
    commodities_list = list(model.commodities.values())
    commodity_names = [c.commodity.value.replace('_', ' ').title() for c in commodities_list]
    
    # Plot 1: Supply/Demand Balance
    ax1 = fig.add_subplot(gs[0, :])
    x = np.arange(len(commodities_list))
    width = 0.35
    supply_vals = [c.supply for c in commodities_list]
    demand_vals = [c.demand for c in commodities_list]
    ax1.bar(x - width/2, supply_vals, width, label='Supply', color='green', alpha=0.7)
    ax1.bar(x + width/2, demand_vals, width, label='Demand', color='red', alpha=0.7)
    ax1.set_ylabel('Volume (commodity-specific units)', fontsize=12)
    ax1.set_title('Supply/Demand Balance Across Commodities', fontsize=14, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(commodity_names)
    ax1.legend()
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Plot 2: Inventory vs Average
    ax2 = fig.add_subplot(gs[1, 0])
    vs_avg = [c.vs_avg_pct for c in commodities_list]
    colors = ['red' if v < 0 else 'green' for v in vs_avg]
    ax2.barh(commodity_names, vs_avg, color=colors, alpha=0.7)
    ax2.axvline(x=0, color='black', linestyle='--', linewidth=1)
    ax2.set_xlabel('% vs Historical Average', fontsize=12)
    ax2.set_title('Inventory Position', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='x')
    
    # Plot 3: Days of Supply
    ax3 = fig.add_subplot(gs[1, 1])
    days_supply = [c.days_supply for c in commodities_list]
    ax3.bar(commodity_names, days_supply, color='steelblue', alpha=0.7)
    ax3.axhline(y=25, color='orange', linestyle='--', linewidth=1, label='25 Days (Reference)')
    ax3.set_ylabel('Days', fontsize=12)
    ax3.set_title('Days of Supply', fontsize=12, fontweight='bold')
    ax3.tick_params(axis='x', rotation=45)
    ax3.legend()
    ax3.grid(True, alpha=0.3, axis='y')
    
    # Plot 4: Market Balance Summary
    ax4 = fig.add_subplot(gs[1, 2])
    balance_counts = {}
    for c in commodities_list:
        balance_counts[c.market_balance.value] = balance_counts.get(c.market_balance.value, 0) + 1
    colors_map = {'tight': 'red', 'balanced': 'yellow', 'oversupplied': 'green'}
    ax4.pie(balance_counts.values(), labels=balance_counts.keys(), autopct='%1.0f%%',
            colors=[colors_map.get(k, 'gray') for k in balance_counts.keys()],
            startangle=90)
    ax4.set_title('Market Balance Distribution', fontsize=12, fontweight='bold')
    
    # Plot 5: Cross-Commodity Network
    ax5 = fig.add_subplot(gs[2, :])
    ax5.axis('off')
    
    # Create relationship matrix
    n_comm = len(commodities_list)
    relationship_strength = np.zeros((n_comm, n_comm))
    
    comm_to_idx = {c.commodity: i for i, c in enumerate(commodities_list)}
    for rel in model.relationships:
        i = comm_to_idx[rel.source]
        j = comm_to_idx[rel.target]
        relationship_strength[i, j] = rel.strength
    
    # Position commodities in a circle
    angles = np.linspace(0, 2*np.pi, n_comm, endpoint=False)
    x_pos = np.cos(angles)
    y_pos = np.sin(angles)
    
    # Draw relationships
    for rel in model.relationships:
        i = comm_to_idx[rel.source]
        j = comm_to_idx[rel.target]
        ax5.annotate('', xy=(x_pos[j], y_pos[j]), xytext=(x_pos[i], y_pos[i]),
                    arrowprops=dict(arrowstyle='->', lw=rel.strength*3, 
                                  alpha=0.6, color='steelblue'))
    
    # Draw commodity nodes
    for i, name in enumerate(commodity_names):
        ax5.scatter(x_pos[i], y_pos[i], s=1000, c='lightblue', 
                   edgecolors='black', linewidth=2, zorder=3)
        ax5.text(x_pos[i], y_pos[i], name, ha='center', va='center',
                fontsize=10, fontweight='bold')
    
    ax5.set_xlim(-1.5, 1.5)
    ax5.set_ylim(-1.5, 1.5)
    ax5.set_title('Cross-Commodity Relationships', fontsize=14, fontweight='bold')
    
    plt.suptitle('Integrated Commodity Fundamental Dashboard', 
                fontsize=16, fontweight='bold', y=0.995)
    plt.show()


# Generate dashboard
create_integrated_dashboard(integrated_model)

## 7. LLM-Powered Integrated Analysis

In [ ]:
# Purpose: Generate comprehensive cross-commodity assessment
# Key Concept: Synthesizing multi-market fundamentals into unified view

class IntegratedAnalyst:
    """LLM-powered integrated commodity analyst."""
    
    def __init__(self, api_key: Optional[str] = None):
        self.llm = ChatAnthropic(
            model="claude-3-5-sonnet-20241022",
            api_key=api_key or os.environ.get("ANTHROPIC_API_KEY"),
            temperature=0.3
        )
        
    def generate_integrated_assessment(self, 
                                      model: IntegratedFundamentalModel) -> str:
        """
        Generate integrated multi-commodity fundamental assessment.
        
        Parameters
        ----------
        model : IntegratedFundamentalModel
            Populated integrated model
            
        Returns
        -------
        str
            Comprehensive assessment
        """
        # Build comprehensive context
        context_parts = ["INTEGRATED COMMODITY FUNDAMENTAL ANALYSIS\n"]
        
        # Individual commodity summaries
        for commodity in model.commodities.values():
            context_parts.append(f"""
{commodity.commodity.value.upper().replace('_', ' ')}:
- Market Balance: {commodity.market_balance.value}
- Supply: {commodity.supply:.1f}, Demand: {commodity.demand:.1f}
- Balance: {commodity.balance:+.1f}
- Inventory: {commodity.inventory:.1f} (vs avg: {commodity.vs_avg_pct:+.1f}%)
- Days Supply: {commodity.days_supply:.1f} days
            """)
        
        # Cross-commodity relationships
        context_parts.append("\nCROSS-COMMODITY RELATIONSHIPS:")
        for rel in model.relationships:
            context_parts.append(
                f"- {rel.source.value} → {rel.target.value}: {rel.description}"
            )
        
        # Calculate derived metrics
        crude = model.commodities[CommodityType.CRUDE_OIL]
        gas = model.commodities[CommodityType.NATURAL_GAS]
        
        associated_gas = model.calculate_associated_gas(crude.supply)
        substitution = model.analyze_substitution(72.0, 2.5)
        
        context_parts.append(f"""
DERIVED METRICS:
- Associated Gas from Oil: {associated_gas:.1f} Bcf/d ({(associated_gas/gas.supply)*100:.1f}% of supply)
- Oil-Gas Substitution: Gas has ${substitution['gas_advantage']:.2f}/MMBtu advantage
        """)
        
        context = "\n".join(context_parts)
        
        # Generate assessment
        prompt = ChatPromptTemplate.from_messages([
            ("system", """You are an expert energy market analyst providing integrated 
fundamental analysis across crude oil, natural gas, and refined products.

Provide a comprehensive assessment including:
1. EXECUTIVE SUMMARY: 2-3 sentence overview of energy complex
2. KEY THEMES: 3-4 major fundamental themes across commodities
3. CROSS-COMMODITY IMPACTS: How fundamentals in one market affect others
4. RELATIVE VALUE: Which commodities appear tight vs oversupplied
5. FORWARD OUTLOOK: 4-8 week view considering all markets
6. KEY RISKS: Top 3 risks to the fundamental outlook
7. TRADING IMPLICATIONS: How to position based on fundamentals

Be specific about numbers and market dynamics."""),
            ("user", "{context}")
        ])
        
        chain = prompt | self.llm
        response = chain.invoke({"context": context})
        
        return response.content


# Generate integrated assessment
analyst = IntegratedAnalyst()
integrated_assessment = analyst.generate_integrated_assessment(integrated_model)

print("=== INTEGRATED FUNDAMENTAL ASSESSMENT ===")
print(integrated_assessment)

## 8. Time Series Integration

In [ ]:
# Purpose: Track integrated fundamentals over time
# Key Concept: Historical fundamental evolution across commodities

def generate_integrated_time_series(weeks: int = 12) -> pd.DataFrame:
    """
    Generate integrated time series for all commodities.
    In production, this aggregates historical EIA data.
    """
    dates = pd.date_range(end=datetime.now(), periods=weeks, freq='W')
    
    data = []
    for i, date in enumerate(dates):
        # Add correlated noise and trends
        market_shock = np.random.normal(0, 0.5)  # Common market factor
        
        data.append({
            'date': date,
            'crude_inventory_vs_avg': -4.0 + i*0.3 + market_shock + np.random.normal(0, 1),
            'gas_inventory_vs_avg': 10.0 - i*0.5 + market_shock + np.random.normal(0, 1.5),
            'gasoline_inventory_vs_avg': 2.5 + market_shock + np.random.normal(0, 0.8),
            'diesel_inventory_vs_avg': -8.5 + i*0.2 + market_shock + np.random.normal(0, 1),
            'crude_days_supply': 21 + np.random.normal(0, 1),
            'gas_days_supply': 28 - i*0.3 + np.random.normal(0, 1.5),
        })
    
    return pd.DataFrame(data)


# Generate time series
ts_integrated = generate_integrated_time_series()

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Plot 1: Inventory positions over time
ax1 = axes[0]
ax1.plot(ts_integrated['date'], ts_integrated['crude_inventory_vs_avg'], 
         marker='o', label='Crude Oil', linewidth=2)
ax1.plot(ts_integrated['date'], ts_integrated['gas_inventory_vs_avg'], 
         marker='s', label='Natural Gas', linewidth=2)
ax1.plot(ts_integrated['date'], ts_integrated['gasoline_inventory_vs_avg'], 
         marker='^', label='Gasoline', linewidth=2)
ax1.plot(ts_integrated['date'], ts_integrated['diesel_inventory_vs_avg'], 
         marker='d', label='Diesel', linewidth=2)
ax1.axhline(y=0, color='black', linestyle='--', linewidth=1)
ax1.fill_between(ts_integrated['date'], 0, ts_integrated['crude_inventory_vs_avg'],
                 where=ts_integrated['crude_inventory_vs_avg']>=0, alpha=0.1, color='green')
ax1.fill_between(ts_integrated['date'], 0, ts_integrated['crude_inventory_vs_avg'],
                 where=ts_integrated['crude_inventory_vs_avg']<0, alpha=0.1, color='red')
ax1.set_ylabel('% vs Historical Average', fontsize=12)
ax1.set_title('Inventory Positions Across Commodities', fontsize=14, fontweight='bold')
ax1.legend(loc='best')
ax1.grid(True, alpha=0.3)

# Plot 2: Days of supply convergence
ax2 = axes[1]
ax2.plot(ts_integrated['date'], ts_integrated['crude_days_supply'],
         marker='o', label='Crude', linewidth=2, color='brown')
ax2.plot(ts_integrated['date'], ts_integrated['gas_days_supply'],
         marker='s', label='Natural Gas', linewidth=2, color='blue')
ax2.axhline(y=25, color='gray', linestyle=':', linewidth=1, label='25 Days Reference')
ax2.set_xlabel('Date', fontsize=12)
ax2.set_ylabel('Days of Supply', fontsize=12)
ax2.set_title('Supply Coverage Trends', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate correlations
corr_matrix = ts_integrated[[
    'crude_inventory_vs_avg', 
    'gas_inventory_vs_avg',
    'gasoline_inventory_vs_avg',
    'diesel_inventory_vs_avg'
]].corr()

print("\nInventory Position Correlations:")
print(corr_matrix.round(2))

## Exercise 4.3: Build Your Own Integrated Model

In [ ]:
# Exercise: Create an integrated model with different commodity mix
#
# Task:
# 1. Add wheat and corn to the integrated model
# 2. Define relationships (corn → ethanol, weather affects both)
# 3. Generate an integrated agricultural-energy assessment
# 4. Visualize cross-sector relationships

# YOUR CODE HERE
# ---------------

# Step 1: Define wheat and corn fundamentals
wheat_fundamental = None  # Create CommodityFundamental

# Step 2: Define corn-ethanol relationship
corn_ethanol_relationship = None  # Create CrossCommodityRelationship

# Step 3: Generate assessment
ag_energy_assessment = None  # Use IntegratedAnalyst

## Summary

### Key Takeaways

1. **Cross-Market Integration**: Commodities don't exist in isolation - model relationships
2. **Associated Production**: Oil production drives associated gas supply (Permian example)
3. **Substitution Economics**: Calculate relative value between competing commodities
4. **Portfolio View**: Assess fundamentals at portfolio level, not just individual markets
5. **LLM Synthesis**: Use LLMs to integrate multi-market fundamentals into coherent narratives

### Module 4 Complete

You've now built:
- **4.1**: Crude oil fundamental models with LLM extraction
- **4.2**: Natural gas storage models with weather integration  
- **4.3**: Integrated multi-commodity fundamental framework

### What's Next

In **Module 5: Signal Generation**, we'll:
- Convert fundamental analysis into discrete trading signals
- Build signal pipelines with confidence scoring
- Backtest LLM-generated signals
- Implement position sizing based on fundamental conviction

### Additional Resources

- [EIA Today in Energy](https://www.eia.gov/todayinenergy/) - Cross-market analysis
- "Energy Trading & Investing" by Davis Edwards
- IEA World Energy Outlook
- BP Statistical Review of World Energy

---

**Practice Exercise**: Build an integrated model using real-time EIA API data for all commodities covered. Update weekly and track how fundamental relationships evolve.